# Runner — Anatomy-Aware Pose Estimation
Notebook **minimale**: tutto il codice vero sta nei moduli `.py` sul repo GitHub.
Qui dentro si fa solo: clone del repo → import → dati → training → valutazione.

**Prima di lanciare:**
1. Settings (pannello a destra) → **Internet: On** (serve per il `git clone`).
2. **Add Input** → aggiungi *COCO 2017 Keypoints* e il dataset *OCHuman* (condiviso).


In [ ]:
# === 1. Codice dal repo GitHub (single source of truth) ===
import os, sys

REPO_URL = "https://github.com/USER/REPO.git"   # <-- METTETE il vostro repo (pubblico)
REPO_DIR = "/kaggle/working/repo"

# --- Repo PRIVATO? Salva un token in Add-ons -> Secrets (nome: GITHUB_TOKEN) e usa: ---
# from kaggle_secrets import UserSecretsClient
# tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
# REPO_URL = f"https://{tok}@github.com/USER/REPO.git"

!rm -rf {REPO_DIR}
!git clone -q {REPO_URL} {REPO_DIR}
sys.path.insert(0, REPO_DIR)


In [ ]:
# === 2. Import dei moduli + seed + check dataset montati ===
from config import *                              # globals, device, set_seed, path
import utils, data, network, train
import evaluation as ev

set_seed(SEED)
print("Device:", device)

# Verifica che i dataset siano montati e che i path in config.py siano giusti.
# Se OCHuman ha uno slug diverso da quello in config.py, CORREGGI config.py e fai push.
print("\n/kaggle/input contiene:")
for d in os.listdir("/kaggle/input"):


In [ ]:
# === 3. Dati COCO ===
from torch.utils.data import DataLoader

train_samples = data.build_samples(COCO_TRAIN_ANN)
val_samples   = data.build_samples(COCO_VAL_ANN)
print(f"train: {len(train_samples)} | val: {len(val_samples)}")

train_dataset = data.COCOKeypointsDataset(train_samples, COCO_TRAIN_IMG, INPUT_SIZE, HEATMAP_SIZE, SIGMA, NUM_KEYPOINTS)
val_dataset   = data.COCOKeypointsDataset(val_samples,   COCO_VAL_IMG,   INPUT_SIZE, HEATMAP_SIZE, SIGMA, NUM_KEYPOINTS)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)


In [ ]:
# === 4. Training (lanciare di sera) ===
import torch

NUM_EPOCHS = 30
model = network.PoseMobileNet(NUM_KEYPOINTS, INPUT_SIZE, pretrained=True).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[15, 25], gamma=0.1)
criterion = train.WeightedMSELoss()

history = train.fit(model, train_loader, val_loader, optimizer, scheduler, criterion,


In [ ]:
# === 5. Valutazione: COCO val + OCHuman zero-shot ===
import torch
model.load_state_dict(torch.load(f"{CHECKPOINT_DIR}/best.pth", map_location=device))

coco_eval, avr_coco = ev.evaluate_on_coco_val(model, val_samples, device)
oc_eval,   avr_oc   = ev.evaluate_on_ochuman(model, device)

LABELS = ["AP","AP.50","AP.75","AP_M","AP_L","AR","AR.50","AR.75","AR_M","AR_L"]
print(f"\n{'metrica':<10}{'COCO val':>12}{'OCHuman':>12}")
print("-"*34)
for i, lab in enumerate(LABELS):
    print(f"{lab:<10}{coco_eval.stats[i]:>12.4f}{oc_eval.stats[i]:>12.4f}")
print(f"\n{'AVR rate':<10}{avr_coco['AVR_pose_rate']:>12.4f}{avr_oc['AVR_pose_rate']:>12.4f}")
